# NB09 The probabilistic nature of finality — Chapter 9

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB09-1: the formula in §11 of the white paper — a direct port of the C code that reproduces the paper's tables (q=0.1, 0.3)
import math

def nakamoto_P(q: float, z: int) -> float:
    """P = 1 - Σ_{k=0}^{z} λ^k e^{-λ}/k! · (1 - (q/p)^{z-k}),  λ = z·q/p   (Nakamoto 2008, p.7)"""
    p = 1.0 - q
    lam = z * (q / p)
    total = 1.0
    for k in range(z + 1):
        poisson = math.exp(-lam)
        for i in range(1, k + 1):
            poisson *= lam / i
        total -= poisson * (1 - (q / p) ** (z - k))
    return total

def gamblers_ruin(q: float, z: int) -> float:
    """First formula of §11, q_z: probability the attacker ever catches up from z blocks behind (ignores the Poisson progress)"""
    p = 1.0 - q
    return 1.0 if p <= q else (q / p) ** z

for q, zs in [(0.1, range(0, 11)), (0.3, range(0, 51, 5))]:
    print(f"q={q}")
    for z in zs:
        print(f"  z={z:<3} P={nakamoto_P(q, z):.7f}   (q_z, gambler's ruin only = {gamblers_ruin(q, z):.7f})")

q=0.1
  z=0   P=1.0000000   (q_z, gambler's ruin only = 1.0000000)
  z=1   P=0.2045873   (q_z, gambler's ruin only = 0.1111111)
  z=2   P=0.0509779   (q_z, gambler's ruin only = 0.0123457)
  z=3   P=0.0131722   (q_z, gambler's ruin only = 0.0013717)
  z=4   P=0.0034552   (q_z, gambler's ruin only = 0.0001524)
  z=5   P=0.0009137   (q_z, gambler's ruin only = 0.0000169)
  z=6   P=0.0002428   (q_z, gambler's ruin only = 0.0000019)
  z=7   P=0.0000647   (q_z, gambler's ruin only = 0.0000002)
  z=8   P=0.0000173   (q_z, gambler's ruin only = 0.0000000)
  z=9   P=0.0000046   (q_z, gambler's ruin only = 0.0000000)
  z=10  P=0.0000012   (q_z, gambler's ruin only = 0.0000000)
q=0.3
  z=0   P=1.0000000   (q_z, gambler's ruin only = 1.0000000)
  z=5   P=0.1773523   (q_z, gambler's ruin only = 0.0144583)
  z=10  P=0.0416605   (q_z, gambler's ruin only = 0.0002090)
  z=15  P=0.0101008   (q_z, gambler's ruin only = 0.0000030)
  z=20  P=0.0024804   (q_z, gambler's ruin only = 0.0000000)
  z=25  P=0.

In [ ]:
# NB09-2: "P < 0.1%" on p. 8 of the white paper — the minimum number of confirmations z for each attacker share q
def min_z(P_func, q, threshold=0.001):
    z = 0
    while P_func(q, z) > threshold:
        z += 1
    return z

print("q      z (Nakamoto formula)")
for q in (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45):
    print(f"{q:.2f}   {min_z(nakamoto_P, q):>4}")

q      z (Nakamoto formula)
0.10      5
0.15      8
0.20     11
0.25     15
0.30     24
0.35     41
0.40     89
0.45    340


In [ ]:
# NB09-3: the exact formula — computed as a finite negative-binomial sum and checked against Rosenfeld Table 1 and Grunspan–Pérez-Marco Table 1 (no external packages)
from math import lgamma, log, exp

def exact_P(q: float, z: int) -> float:
    """While the honest chain mines z blocks, the attacker's block count k ~ NegBin; if k<z it catches up with (q/p)^(z-k), if k>=z it already has.
    The binomial coefficients get large, so compute in logs: log C(k+z-1, k) = lgamma(k+z) - lgamma(k+1) - lgamma(z)."""
    if z == 0: return 1.0
    p = 1.0 - q
    if q >= p: return 1.0
    caught, mass = 0.0, 0.0
    for k in range(z):
        log_pk = lgamma(k + z) - lgamma(k + 1) - lgamma(z) + z * log(p) + k * log(q)
        pk = exp(log_pk)
        mass += pk
        caught += pk * (q / p) ** (z - k)
    return caught + (1.0 - mass)

print("Rosenfeld (2014) Table 1, q=10% row: 20%, 5.600%, 1.712%, 0.546%, 0.178%, 0.059%, 0.020%, 0.007%, 0.002%, 0.001%")
print("This book's finite sum, q=10% row:  ", ", ".join(f"{100*exact_P(0.10, n):.3f}%" for n in range(1, 11)))
print()
print("Grunspan–Pérez-Marco (2018) Table 1: z needed for P<0.1% —  q: 0.10 0.15 0.20 0.25 0.30 0.35 0.40 0.45")
print("  the paper's z (exact formula) : 6 9 13 20 32 58 133 539")
print("  this book's finite sum        :", " ".join(str(min_z(exact_P, q)) for q in (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45)))
print("  the paper's z_SN (Nakamoto)   : 5 8 11 15 24 41 81 340   ← the 81 at q=0.40 differs from the 89 on p. 8 of the white paper and in NB09-2 above (see the text below)")
print()
print(f"{'q':>5} {'z':>4} {'Nakamoto':>10} {'exact':>10} {'ratio':>6}")
for q, z in ((0.1, 1), (0.1, 3), (0.1, 6), (0.3, 5), (0.3, 10), (0.3, 24), (0.45, 40), (0.45, 340)):
    a, b = nakamoto_P(q, z), exact_P(q, z)
    print(f"{q:>5.2f} {z:>4} {a:>10.7f} {b:>10.7f} {b/a:>6.2f}")

Rosenfeld (2014) Table 1, q=10% row: 20%, 5.600%, 1.712%, 0.546%, 0.178%, 0.059%, 0.020%, 0.007%, 0.002%, 0.001%
This book's finite sum, q=10% row:   20.000%, 5.600%, 1.712%, 0.546%, 0.178%, 0.059%, 0.020%, 0.007%, 0.002%, 0.001%

Grunspan–Pérez-Marco (2018) Table 1: z needed for P<0.1% —  q: 0.10 0.15 0.20 0.25 0.30 0.35 0.40 0.45
  the paper's z (exact formula) : 6 9 13 20 32 58 133 539
  this book's finite sum        : 6 9 13 20 32 58 133 539
  the paper's z_SN (Nakamoto)   : 5 8 11 15 24 41 81 340   ← the 81 at q=0.40 differs from the 89 on p. 8 of the white paper and in NB09-2 above (see the text below)

    q    z   Nakamoto      exact  ratio
 0.10    1  0.2045873  0.2000000   0.98
 0.10    3  0.0131722  0.0171200   1.30
 0.10    6  0.0002428  0.0005914   2.44
 0.30    5  0.1773523  0.1976173   1.11
 0.30   10  0.0416605  0.0651067   1.56
 0.30   24  0.0008106  0.0040007   4.94
 0.45   40  0.3454201  0.3713860   1.08
 0.45  340  0.0009947  0.0089679   9.02


In [ ]:
# NB09-4: Monte Carlo — do not take the formula on trust; run the race yourself (fixed seed)
import random

def simulate(q: float, z: int, trials: int, seed: int = 20261025) -> float:
    """The attacker mines too until the honest chain has mined z blocks (probability q per block). After that, success if the gap falls to 0 or below; cut off as failure once it exceeds z+200."""
    rng = random.Random(seed)
    p = 1.0 - q
    wins = 0
    for _ in range(trials):
        honest, attacker = 0, 0
        while honest < z:                        # while the merchant waits for z confirmations
            if rng.random() < q: attacker += 1
            else: honest += 1
        gap = honest - attacker
        while 0 < gap < z + 200:                 # the race after that
            gap += -1 if rng.random() < q else 1
        if gap <= 0: wins += 1
    return wins / trials

for q, z, n in ((0.1, 3, 200_000), (0.1, 6, 200_000), (0.3, 10, 50_000)):
    mc, ex, na = simulate(q, z, n), exact_P(q, z), nakamoto_P(q, z)
    se = (ex * (1 - ex) / n) ** 0.5
    print(f"q={q} z={z:>2} n={n:>7,}: Monte Carlo {mc:.5f} | exact {ex:.5f} | Nakamoto {na:.5f} | MC standard error ≈ {se:.5f}")

q=0.1 z= 3 n=200,000: Monte Carlo 0.01728 | exact 0.01712 | Nakamoto 0.01317 | MC standard error ≈ 0.00029
q=0.1 z= 6 n=200,000: Monte Carlo 0.00061 | exact 0.00059 | Nakamoto 0.00024 | MC standard error ≈ 0.00005
q=0.3 z=10 n= 50,000: Monte Carlo 0.06414 | exact 0.06511 | Nakamoto 0.04166 | MC standard error ≈ 0.00110


In [ ]:
# NB09-5: invariance to absolute hash rate — only the share q enters the formula
print("attacker hash rate a, honest hash rate h  →  q = a/(a+h),  P(z=6)")
for a, h in ((1, 9), (1e6, 9e6), (1e20, 9e20), (3e20, 7e20)):
    q = a / (a + h)
    print(f"  a={a:.0e} H/s, h={h:.0e} H/s → q={q:.2f}, exact P(6)={exact_P(q, 6):.6f}, Nakamoto P(6)={nakamoto_P(q, 6):.6f}")

attacker hash rate a, honest hash rate h  →  q = a/(a+h),  P(z=6)
  a=1e+00 H/s, h=9e+00 H/s → q=0.10, exact P(6)=0.000591, Nakamoto P(6)=0.000243
  a=1e+06 H/s, h=9e+06 H/s → q=0.10, exact P(6)=0.000591, Nakamoto P(6)=0.000243
  a=1e+20 H/s, h=9e+20 H/s → q=0.10, exact P(6)=0.000591, Nakamoto P(6)=0.000243
  a=3e+20 H/s, h=7e+20 H/s → q=0.30, exact P(6)=0.156450, Nakamoto P(6)=0.132111
